In [3]:
from utils.log_config import logs_config
from dotenv import load_dotenv
import requests
import os
import polars as pl

In [4]:
load_dotenv("../.env", override=True)

API_KEY = os.getenv("TFL_API_KEY")

### BikePoint endpoint

BikePoint
 Gets all bike point locations. The Place object has an addtionalProperties array which contains the nbBikes, nbDocks and nbSpaces numbers which give the status of the BikePoint. A mismatch in these numbers i.e. nbDocks - (nbBikes + nbSpaces) != 0 indicates broken docks.

| Name | Required | Type | Description |
|------|----------|------|-------------|
|  id  |  false   | string | A unique identifier |
| url |false |string |The unique location of this resource |
| commonName | false | string | A human readable name |
| distance | false | number (double) |The distance of the place from its search point, if this is the result of a geographical search, otherwise zero |
| placeType | false | string |The type of Place. See /Place/Meta/placeTypes for possible values |
| additionalProperties | false | Tfl.Api.Presentation.Entities.AdditionalProperties[] |A bag of additional key/value pairs with extra information about this place |
| children | false | Tfl.Api.Presentation.Entities.Place[] | |
| childrenUrls | false | string[] | |
| lat | false | number (double) | WGS84 latitude of the location |
| lon | false | number (double) | WGS84 longitude of the location |

In [75]:
request_url = "https://api.tfl.gov.uk/BikePoint/"

search_response = requests.get(request_url)

search_response.raise_for_status()

data = search_response.json()

In [76]:
bikepoint_facts = {}
bikepoint_dim = {}

for item in data:
    # 1. Convert the list of properties into one flat dictionary
    # Example: {"NbBikes": "7", "NbEmptyDocks": "9", ...}
    props_map = {p["key"]: p for p in item["additionalProperties"]}

    # Helper to get value safely
    def get_val(key): return props_map.get(key, {}).get("value")

    # Helper to get modified date safely
    def get_mod(key): return props_map.get(key, {}).get("modified")
    
    # 2. Assign all the data you want to the ID
    bikepoint_dim[item["id"]] = {
        "commonName": item["commonName"],
        "installed": get_val("Installed"),
        "locked": get_val("Locked"),
        "removal_date": get_val("RemovalDate"),
        "temporary": get_val("Temporary"),
        "lat": item["lat"],
        "lon": item["lon"]
    }

    # 3. Assign all the data you want to the ID
    bikepoint_facts[item["id"]] = {
        "total_docks": get_val("NbDocks"),
        "total_docks_modified": get_mod("NbDocks"),

        "emptydocks": get_val("NbEmptyDocks"),
        "emptydocks_modified": get_mod("NbEmptyDocks"),
        
        "standardBikes": get_val("NbStandardBikes"),
        "standardBikes_modified": get_mod("NbStandardBikes"),
        
        "ebikes": get_val("NbEBikes"),
        "ebikes_modified": get_mod("NbEBikes"),
        
        "nbBikes": get_val("NbBikes"),
        "nbBikes_modified": get_mod("NbBikes")
    }

In [77]:
bikepoint_dim

{'BikePoints_1': {'commonName': 'River Street , Clerkenwell',
  'installed': 'true',
  'locked': 'false',
  'removal_date': '',
  'temporary': 'false',
  'lat': 51.529163,
  'lon': -0.10997},
 'BikePoints_2': {'commonName': 'Phillimore Gardens, Kensington',
  'installed': 'true',
  'locked': 'false',
  'removal_date': '',
  'temporary': 'false',
  'lat': 51.499606,
  'lon': -0.197574},
 'BikePoints_3': {'commonName': 'Christopher Street, Liverpool Street',
  'installed': 'true',
  'locked': 'false',
  'removal_date': '',
  'temporary': 'false',
  'lat': 51.521283,
  'lon': -0.084605},
 'BikePoints_4': {'commonName': "St. Chad's Street, King's Cross",
  'installed': 'true',
  'locked': 'false',
  'removal_date': '',
  'temporary': 'false',
  'lat': 51.530059,
  'lon': -0.120973},
 'BikePoints_5': {'commonName': 'Sedding Street, Sloane Square',
  'installed': 'true',
  'locked': 'false',
  'removal_date': '',
  'temporary': 'false',
  'lat': 51.49313,
  'lon': -0.156876},
 'BikePoints_6'

In [78]:
bikepoint_facts

{'BikePoints_1': {'total_docks': '19',
  'total_docks_modified': '2026-03-24T23:55:33.56Z',
  'emptydocks': '10',
  'emptydocks_modified': '2026-03-24T23:55:33.56Z',
  'standardBikes': '2',
  'standardBikes_modified': '2026-03-24T23:55:33.56Z',
  'ebikes': '1',
  'ebikes_modified': '2026-03-24T23:55:33.56Z',
  'nbBikes': '3',
  'nbBikes_modified': '2026-03-24T23:55:33.56Z'},
 'BikePoints_2': {'total_docks': '37',
  'total_docks_modified': '2026-03-24T21:27:19.803Z',
  'emptydocks': '30',
  'emptydocks_modified': '2026-03-24T21:27:19.803Z',
  'standardBikes': '3',
  'standardBikes_modified': '2026-03-24T21:27:19.803Z',
  'ebikes': '1',
  'ebikes_modified': '2026-03-24T21:27:19.803Z',
  'nbBikes': '4',
  'nbBikes_modified': '2026-03-24T21:27:19.803Z'},
 'BikePoints_3': {'total_docks': '32',
  'total_docks_modified': '2026-03-24T23:35:24.967Z',
  'emptydocks': '26',
  'emptydocks_modified': '2026-03-24T23:35:24.967Z',
  'standardBikes': '1',
  'standardBikes_modified': '2026-03-24T23:35:2